# Network Security Traffic Analyzer — Public Demo
Synthetic-only demonstration. No private course materials are included.


### Setup
Local Jupyter: extract archive and run. Google Colab: if project files are missing, select the public release ZIP when prompted.


In [ ]:
from pathlib import Path
import sys, zipfile
base=Path.cwd()
root=next((p for p in [base,base/'Network_Security_Traffic_Analyzer_Public_Release',Path('/content/Network_Security_Traffic_Analyzer_Public_Release')] if (p/'analyzer_v04.py').is_file() and (p/'synthetic_data').is_dir()),None)
if root is None:
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Extract the public release ZIP and launch the notebook from the project folder.')
    print('Select Network_Security_Traffic_Analyzer_Public_Release.zip')
    uploaded=files.upload()
    archives=[name for name in uploaded if name.lower().endswith('.zip')]
    if len(archives)!=1: raise ValueError('Select exactly one public release ZIP')
    with zipfile.ZipFile(archives[0]) as z:
        for member in z.infolist():
            target=(base/member.filename).resolve()
            if not target.is_relative_to(base.resolve()): raise ValueError('Unsafe ZIP path')
        z.extractall(base)
    root=base/'Network_Security_Traffic_Analyzer_Public_Release'
    if not (root/'analyzer_v04.py').is_file(): raise FileNotFoundError('Wrong ZIP: analyzer_v04.py missing')
sys.path.insert(0,str(root))
print('Project:',root)


In [ ]:
from analyzer_v04 import extract_tcpdump, extract_wireshark, summarize, save
sources={'dns_icmp':('dns_icmp.txt',extract_tcpdump),'http':('http.txt',extract_tcpdump),'syn':('syn.txt',extract_wireshark)}
evidence={}
for case,(filename,parser) in sources.items():
    events,rejected=parser((root/'synthetic_data'/filename).read_text(encoding='utf-8'),filename)
    evidence[case]={'events':events,'rejected':rejected}
    print(case,'parsed:',len(events),'rejected:',len(rejected))


In [ ]:
report=summarize(evidence)
report['evidence_origin']='synthetic_demo'
report['limitations']=['Synthetic demonstration, not evidence of a real incident.']
for case,item in report['reports'].items():
    print(case,item['counts'],[(f['id'],f['classification']) for f in item['findings']])
output=root/'reports'/'synthetic_demo_report.json'
output.parent.mkdir(exist_ok=True)
save(report,output)
print('Saved:',output)


### Evidence quality
All included inputs are synthetic examples. Detection of a SYN burst is suspicious, not proof of attack intent or impact.
### Tests
Run `python -m pytest -q` from the project folder.
